In [2]:
import numpy as np

def solve_three_temp_harmonic(
    m: int,
    Teq_hat: complex,
    *,
    gamma_g: float = 1.0,
    beta_gs: float,
    beta_gb: float,
    beta_sg: float,
    beta_bg: float,
    beta_rs: float,
    beta_rb: float,
):
    """
    Solve the Fourier-harmonic (e^{i m phi}) 3-temperature system:

        dTg/dphi = -(1/gamma_g) [ (Tg-Ts)/beta_gs + (Tg-Tb)/beta_gb ]
        dTs/dphi = + (Tg-Ts)/beta_sg - (Ts-Teq)/beta_rs
        dTb/dphi = + (Tg-Tb)/beta_bg - (Tb-Teq)/beta_rb

    With ansatz: T(φ) = Re{ T_hat * exp(i m φ) }, so d/dφ -> i m.

    Returns:
        Tg_hat, Ts_hat, Tb_hat (complex)
    """
    im = 1j * m

    # Build A @ [Tg, Ts, Tb]^T = b
    A = np.array([
        [im + (1.0/(gamma_g*beta_gs)) + (1.0/(gamma_g*beta_gb)),   -(1.0/(gamma_g*beta_gs)),               -(1.0/(gamma_g*beta_gb))],
        [-(1.0/beta_sg),                                          im + (1.0/beta_sg) + (1.0/beta_rs),      0.0],
        [-(1.0/beta_bg),                                          0.0,                                    im + (1.0/beta_bg) + (1.0/beta_rb)],
    ], dtype=complex)

    b = np.array([
        0.0,
        (1.0/beta_rs) * Teq_hat,
        (1.0/beta_rb) * Teq_hat,
    ], dtype=complex)

    Tg_hat, Ts_hat, Tb_hat = np.linalg.solve(A, b)
    return Tg_hat, Ts_hat, Tb_hat


def phase_shift_per_m(z: complex, m: int) -> float:
    """Return phase shift Δφ = arg(z)/m in radians."""
    return np.angle(z) / m


# ---- Example usage (plug in your betas) ----
if __name__ == "__main__":
    m = 2
    Teq_hat = 1.0 + 0.0j

    Tg_hat, Ts_hat, Tb_hat = solve_three_temp_harmonic(
        m=m,
        Teq_hat=Teq_hat,
        gamma_g=1.4,
        beta_gs=1e-2,   # gas -> small dust
        beta_gb=1e0,    # gas -> big dust
        beta_sg=1e-2,   # small dust -> gas
        beta_bg=1e0,    # big dust -> gas
        beta_rs=1e-1,   # small dust radiative relaxation to Teq
        beta_rb=1e-1,   # big dust radiative relaxation to Teq
    )

    print("Tg_hat =", Tg_hat)
    print("Ts_hat =", Ts_hat)
    print("Tb_hat =", Tb_hat)

    print("phase shifts (rad):",
          phase_shift_per_m(Tg_hat, m),
          phase_shift_per_m(Ts_hat, m),
          phase_shift_per_m(Tb_hat, m))
    print("amplitudes:", abs(Tg_hat), abs(Ts_hat), abs(Tb_hat))


Tg_hat = (0.8170216311460229-0.3962911753378197j)
Ts_hat = (0.8268324279195876-0.3752980217238286j)
Tb_hat = (0.945557244735445-0.2079459695280645j)
phase shifts (rad): -0.2258055594823851 -0.2130456054096726 -0.10823650429908853
amplitudes: 0.9080589415952791 0.9080200818094384 0.968152895628915


In [ ]:
beta_rg = 1e-1
coagulation_factor = 1e-2
beta_rs = 1e-1